# Day 2: H3 hands-on

| v3 (old) | v4 (use) |
|---|---|
| `geo_to_h3` | `latlng_to_cell` |
| `h3_to_geo` | `cell_to_latlng` |
| `k_ring` | `grid_disk` |
| `hex_ring` | `grid_ring` |
| `h3_to_geo_boundary` | `cell_to_boundary` |
| `polyfill` | `polygon_to_cells` / `h3shape_to_cells` |
| `h3_distance` | `grid_distance` |

In [1]:
import h3, math, pandas as pd
OFFICE = (43.2380, 76.9450)
cell9 = h3.latlng_to_cell(*OFFICE, 9)
print(cell9, h3.cell_to_latlng(cell9), h3.get_resolution(cell9))

8920e60ae93ffff (43.237485465509465, 76.94514452077331) 9


In [ ]:
rows = []
for res in range(4, 11):
    cell = h3.latlng_to_cell(*OFFICE, res)
    avg_area_km2 = h3.average_hexagon_area(res, unit='km^2')
    cell_area_km2 = h3.cell_area(cell, unit='km^2')
    avg_edge_length_km = h3.average_hexagon_edge_length(res, unit='km')
    rows.append({"res": res, "cell": cell, "avg_area_km2": avg_area_km2, "cell_area_km2": cell_area_km2, "avg_edge_length_km": avg_edge_length_km})

cheat = pd.DataFrame(rows)
cheat["use"] = ["", "", "demand by district", "zones / OD", "street level", "stops / POI", "exact points"]
assert abs(cheat.loc[cheat.res == 8, "avg_area_km2"].item() - 0.737) < 0.02
cheat.round(4)

,res,cell,avg_area_km2,cell_area_km2,avg_edge_length_km,use
0,4,8420e61ffffffff,1770.3477,2114.3662,26.0718,
1,5,8520e60bfffffff,252.9039,301.9423,9.8541,
2,6,8620e60afffffff,36.1291,43.1372,3.7245,demand by district
3,7,8720e60a1ffffff,5.1613,6.1634,1.4065,zones / OD
4,8,8820e60ae9fffff,0.7373,0.8804,0.5314,street level
5,9,8920e60ae93ffff,0.1053,0.1258,0.2008,stops / POI
6,10,8a20e60ae907fff,0.0150,0.0180,0.0759,exact points


In [7]:
disk2, ring2 = h3.grid_disk(cell9, 2), h3.grid_ring(cell9, 2)
assert len(disk2) == 1 + 3 * 2 * 3 == 19 and len(ring2) == 12
far = h3.latlng_to_cell(43.2600, 76.9450, 9)
print("grid_distance до точки 2.4 км севернее:", h3.grid_distance(cell9, far))

grid_distance до точки 2.4 км севернее: 7


## Задание 3 — полигон → ячейки, compact, родители/дети

1. Постройте квадрат ~3×3 км вокруг офиса как `h3.LatLngPoly` и заполните его ячейками res 9
   (`h3.h3shape_to_cells`).
2. `compact_cells` — сколько ячеек стало? Почему меньше?
3. Возьмите родителя `cell9` на res 7 и всех его детей на res 9. Проверьте: `cell9` среди детей,
   но **объединение границ детей ≠ граница родителя** (сравните площади: сумма площадей детей
   против площади родителя — они близки, но форма другая). Ответьте: почему?

In [10]:
dlat, dlon = 0.0135, 0.0185          # ~1.5 км
poly = h3.LatLngPoly([(OFFICE[0]-dlat, OFFICE[1]-dlon), (OFFICE[0]-dlat, OFFICE[1]+dlon),
                      (OFFICE[0]+dlat, OFFICE[1]+dlon), (OFFICE[0]+dlat, OFFICE[1]-dlon)])

cells9 = h3.h3shape_to_cells(poly, 9)
compact = h3.compact_cells(cells9)
parent7 = h3.cell_to_parent(cell9, 7)
children9 = h3.cell_to_children(parent7, 9)

print(len(cells9), "ячеек res9 ->", len(compact), "после compact")
assert len(compact) < len(cells9)
assert cell9 in children9 and len(children9) == 49 
sum_children = sum(h3.cell_area(c, unit="km^2") for c in children9)
print(f"родитель {h3.cell_area(parent7, unit='km^2'):.4f} км², сумма детей {sum_children:.4f} км²")

72 ячеек res9 -> 42 после compact
родитель 6.1628 км², сумма детей 6.1628 км²


In [11]:
import folium, os
m = folium.Map(location=OFFICE, zoom_start=15, tiles="cartodbpositron")
for c in disk2:
    folium.Polygon(h3.cell_to_boundary(c), color="#C4561D" if c == cell9 else "#1F5E96",
                   weight=1, fill=True, fill_opacity=0.15, tooltip=c).add_to(m)
os.makedirs("m01", exist_ok=True); m.save("m01/h3_disk.html"); m

c:\Users\ikhairat\DS-GIS-senior\.venv\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
